Convolución
-----------


In [ ]:
import numpy as np
from numpy import linspace, arange, sin, cos, exp, pi, sinc
from bokeh.models import CustomJS
from bokeh.layouts import Column, Row
from bokeh.models.widgets import Slider
from bokeh.models.sources import ColumnDataSource
from bokeh.plotting import figure, output_notebook, output_file, show
output_notebook()

## Representación en tiempo "continuo"
dt = 1e-2
Tend = 5
t = arange(-Tend, Tend, dt)
f0 = 1
# Señales de prueba 'x' e 'y'
x0 = np.sin(t)/t
y0 = [1/2 if (np.abs(v) < 1) else 0 for v in t]
x = x0
y = [1/2 if (np.abs(v+10) < 1) else 0 for v in t]
# Resultado de la convolución
xy = np.convolve(x0, y0, 'same')
z = np.zeros(len(x))
# Origen de datos para las gráficas
source = ColumnDataSource(data=dict(t=t, x0=x0, y0=y0, x=x, y=y, xy=xy, z=z, x_=y0))
# Gráfica de la señal x
p1 = figure(min_width=300, min_height=300)
p1.title.text = 'x(t)'
p1.title.align = 'center'
p1.title.text_font_size = '25px'
p1.line('t', 'x0', source=source, line_width=2, line_color='orange')
# Gráfica de la señal y
p2 = figure(min_width=300, min_height=300, title='y(t)')
p2.title.text = 'y(t)'
p2.title.align = 'center'
p2.title.text_font_size = '25px'
p2.line('t', 'y0', source=source, line_width=2, line_color='orange')
# Gráfica de la señal z
p3 = figure(min_width=600, min_height=300, title='(x*y)(t)')
p3.title.text = '(x*y)(t)'
p3.title.align = 'center'
p3.title.text_font_size = '25px'
p3.line('t', 'x', source=source, line_width=2, line_color='orange')
p3.line('t', 'y', source=source, line_width=2)
p3.line('t', 'z', source=source, line_width=2, line_color='red')
p3.varea_stack(['x_'], x='t', source=source, alpha=0.4, color='orange')
p3.varea_stack(['y'], x='t', source=source, alpha=0.3)
# Control de 'tau' para mostrar el resultado de la convolución
tau_slider = Slider(start=-Tend, end=Tend, value=-Tend, step=.1, title="t")
# Actualiza la gráfica al modificar el valor de 'tau'
callback = CustomJS(args=dict(source=source, tau_slider=tau_slider, dt=dt, h=2),
                    code="""
    const data = source.data;
    const tau = tau_slider.value;
    const t = data['t'];
    const x = data['x'];
    const x_ = data['x_'];
    const y = data['y'];
    const xy = data['xy'];
    const z = data['z'];

    for (var i = 0; i < t.length; i++) {
        y[i] = (Math.abs(t[i]-tau) < 1) ? 1/2 : 0;
        z[i] = (t[i] < tau) ? xy[i]*dt : 0;
        x_[i] = (y[i] != 0) ? x[i] : 0;
    }
    source.change.emit();
""")
tau_slider.js_on_change('value', callback)
# Exporta a una página web
output_file('convolucion.html')
show(Column(Row(p1, p2), p3, tau_slider))



Loading BokehJS ...

In [1]:
import numpy as np
from numpy import arange
from bokeh.models import CustomJS, ColumnDataSource, Slider, FreehandDrawTool
from bokeh.layouts import Column, Row
from bokeh.plotting import figure, output_file, show

continuo = False
# --- Configuración de Ejes ---
dt = 1e-2 if continuo else 0.1
Tend = 5
t = arange(-Tend, Tend, dt)
N = len(t)
# --- Señales de pruesba ---
x = np.sin(2*t)/(2*t)
y = np.array([1 if (np.abs(v) < 1/2) else 0 for v in t])
z = np.convolve(x, y, 'same')
# --- Fuentes de Datos ---
# source_x: dibujo de la primera señal
source_x = ColumnDataSource(data=dict(xs=[t], ys=[x])) 
# source_y: dibujo de la segunda señal
source_y = ColumnDataSource(data=dict(xs=[t], ys=[y]))
# source_calc: datos para la visualización de la convolución
source_calc = ColumnDataSource(data=dict(
    t=t, 
    x_fijada=x,
    y_invertida=np.zeros(N), 
    resultado=np.zeros(N),
    area_overlap=np.zeros(N)
))

# --- Creación de los Paneles de Dibujo ---
def crear_panel_dibujo(titulo, color, src):
    p = figure(width=600, height=400, title=titulo, x_range=(-Tend, Tend), y_range=(-1.5, 1.5),
               tools="pan,wheel_zoom,reset")
    renderer = p.multi_line('xs', 'ys', source=src, line_width=3, line_color=color, dash='dotted')
    p.scatter('xs', 'ys', source=src, size=10, color='red', marker="circle")

    draw_tool = FreehandDrawTool(renderers=[renderer], num_objects=1)
    p.add_tools(draw_tool)
    p.toolbar.active_drag = draw_tool
    return p

p1 = crear_panel_dibujo("Dibuja x(t)", "orange", source_x)
p2 = crear_panel_dibujo("Dibuja y(t)", "green", source_y)

# --- Gráfica de Resultado ---
p3 = figure(width=1200, height=400, title="Convolución (x * y)(t)", x_range=(-Tend, Tend))
if continuo:
  p3.line('t', 'x_fijada', source=source_calc, line_width=2, color='orange', legend_label="x(\u03C4)")
  p3.line('t', 'y_invertida', source=source_calc, line_width=2, color='green', legend_label="y(t-\u03C4)")
  p3.line('t', 'resultado', source=source_calc, line_width=4, color='red', legend_label="(x*y)(t)")
else:
  p3.scatter('t', 'x_fijada', source=source_calc, line_width=2, color='orange', legend_label="x(\u03C4)")
  p3.scatter('t', 'y_invertida', source=source_calc, line_width=2, color='green', legend_label="y(t-\u03C4)")
  p3.scatter('t', 'resultado', source=source_calc, line_width=4, color='red', legend_label="(x*y)(t)")
p3.varea('t', 0, 'x_fijada', source=source_calc, color='orange', alpha=0.3)
p3.varea('t', 0, 'y_invertida', source=source_calc, color='green', alpha=0.3)
p3.varea('t', 0, 'resultado', source=source_calc, color='red', alpha=0.3)

# --- Lógica de Procesamiento en JavaScript ---
callback_code = """
    const data = source_calc.data;
    const dx = source_x.data;
    const dy = source_y.data;
    const tau = tau_slider.value;
    const t = data['t'];
    const N = t.length;

    // Función auxiliar para proyectar el dibujo manual al eje de tiempo
    function proyectar(draw_data, target_array) {
        const xs = draw_data['xs'][0] || [];
        const ys = draw_data['ys'][0] || [];
        target_array.fill(0);
        for (let i = 0; i < N; i++) {
            for (let j = 0; j < xs.length - 1; j++) {
                if (t[i] >= xs[j] && t[i] <= xs[j+1]) {
                    target_array[i] = ys[j];
                    break;
                }
            }
        }
    }

    function invertir(x, y) {
        for(let i=0; i<N; i++) {
            y[i] = x[N-1-i];
        }
    }

    function desplazar(x, y) {
        let L = Math.round(tau / dt);  
        for (let i=0; i<N; i++) {
            let j = i - L;
            y[i] = (j >= 0 && j < N) ? x[j] : 0;
        }
    }

    function convolucion(t, x, y, z) {
        let L = Math.round((tau - t[0])/dt);
        let i_min = Math.round(t[0]/dt);

        for(let i=0; i<L; i++) {
            z[i] = 0;
            let j_min = Math.max(0, i-i_min-N+1), j_max = Math.min(N, i-i_min+1);
            for(let j=j_min; j<j_max; j++) {
               z[i] += x[j]*y[i-i_min-j]*dt;
            }
        }
    }

    // 1. Obtener formas actuales
    const x = new Float64Array(N);
    const y = new Float64Array(N);
    proyectar(dx, x);
    proyectar(dy, y);
    
    // 2. Calcular y(t - tau) -> Inversión y desplazamiento
    const y_rev = new Float64Array(N);
    const y_inv = new Float64Array(N);
    const z_res = new Float64Array(N);
    invertir(y, y_rev);
    desplazar(y_rev, y_inv);
    convolucion(t, x, y, z_res);
    
    data['x_fijada'].set(x);
    data['y_invertida'].set(y_inv);
    data['resultado'].set(z_res);
    source_calc.change.emit();
"""

tau_slider = Slider(start=-Tend, end=Tend, value=-Tend, step=0.1, title="Desplazamiento (t)", sizing_mode='stretch_width')
callback = CustomJS(args=dict(source_calc=source_calc, source_x=source_x, source_y=source_y, dt=dt, tau_slider=tau_slider),
                    code=callback_code)
tau_slider.js_on_change('value', callback)
# Actualizar al terminar de dibujar en cualquiera de los dos
source_x.js_on_change('data', callback)
source_y.js_on_change('data', callback)

# --- Renderizado ---
layout = Column(Row(p1, p2), Column(p3, tau_slider))
if continuo:
    output_file("html/T1_Convolucion continua.html", title="PdS: Convolución continua")
else:
    output_file("html/T1_Convolucion discreta.html", title="PdS: Convolución discreta")
show(layout)